# Forecasting del mercato del lavoro italiano

Previsione di un indicatore del mercato del lavoro per regione a partire dal layer **gold** del warehouse (popolato dal DAG Airflow `mercato_lavoro_istat`).

Metodo: ARIMA (statsmodels) con backtest sugli ultimi periodi disponibili, valutato con MAPE/RMSE.

**Nota sui dati**: il canale Eurostat fornisce serie **annuali** 2015-2025 (11 punti per regione) — poche osservazioni per un modello di serie storiche, quindi qui l'obiettivo è mostrare il metodo (backtest, metriche, previsione con intervallo) più che ottenere previsioni ad alta confidenza. Con più anni di storico il forecasting diventerebbe più robusto.

**Prerequisito**: il DAG deve aver completato almeno un run con successo (`gold.fact_mercato_lavoro` non vuota).

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sqlalchemy import create_engine
from statsmodels.tsa.arima.model import ARIMA

from src.config import warehouse_connection_string

engine = create_engine(warehouse_connection_string())

## Caricamento dati dal layer gold

In [2]:
fact = pd.read_sql('SELECT * FROM gold.fact_mercato_lavoro', engine)
dim_territorio = pd.read_sql('SELECT * FROM gold.dim_territorio', engine)
dim_tempo = pd.read_sql('SELECT * FROM gold.dim_tempo', engine)

df = fact.merge(dim_territorio, on='cod_territorio').merge(dim_tempo, on='periodo')
df.head()

,indicatore,cod_territorio,periodo,sesso,classe_eta,valore,territorio,ripartizione,anno
0,tasso_disoccupazione,Abruzzo,2015,T,Y15-74,12.6,Abruzzo,Sud,2015
1,tasso_disoccupazione,Abruzzo,2016,T,Y15-74,12.1,Abruzzo,Sud,2016
2,tasso_disoccupazione,Abruzzo,2017,T,Y15-74,11.7,Abruzzo,Sud,2017
3,tasso_disoccupazione,Abruzzo,2018,T,Y15-74,10.8,Abruzzo,Sud,2018
4,tasso_disoccupazione,Abruzzo,2019,T,Y15-74,11.1,Abruzzo,Sud,2019


## Selezione serie storica

Cambia `INDICATORE` e `TERRITORIO` per esplorare altre combinazioni (usa `df['indicatore'].unique()` / `df['territorio'].unique()` per vedere le opzioni disponibili — con il canale Eurostat sono le 20 regioni, non c'è una riga 'Italia' nazionale).

In [3]:
INDICATORE = 'tasso_disoccupazione'
TERRITORIO = 'Lombardia'
SESSO = 'T'
CLASSE_ETA = df[df['indicatore'] == INDICATORE]['classe_eta'].iloc[0]

serie = (
    df[(df['indicatore'] == INDICATORE) & (df['territorio'] == TERRITORIO)
       & (df['sesso'] == SESSO) & (df['classe_eta'] == CLASSE_ETA)]
    .sort_values('periodo')
    .set_index('periodo')['valore']
)
serie.index = pd.to_datetime(serie.index.astype(str), format='%Y')
serie = serie.asfreq('YS')
px.line(serie, title=f'{INDICATORE} - {TERRITORIO}', markers=True)

## Backtest: train/test split sugli ultimi periodi

In [4]:
N_TEST = 3  # ultimi 3 anni tenuti come test set (serie corta: pochi punti disponibili)
train, test = serie.iloc[:-N_TEST], serie.iloc[-N_TEST:]

model = ARIMA(train, order=(1, 1, 0))
fit = model.fit()

forecast_test = fit.get_forecast(steps=N_TEST).predicted_mean
forecast_test.index = test.index

mape = mean_absolute_percentage_error(test, forecast_test)
rmse = mean_squared_error(test, forecast_test) ** 0.5
print(f'MAPE: {mape:.2%}  |  RMSE: {rmse:.2f} punti percentuali')

MAPE: 38.78%  |  RMSE: 1.38 punti percentuali


In [5]:
confronto = pd.concat([
    train.rename('storico'),
    test.rename('reale (test)'),
    forecast_test.rename('previsto (test)'),
], axis=1)
px.line(confronto, title=f'Backtest {INDICATORE} - {TERRITORIO} (MAPE {mape:.1%})', markers=True)

## Previsione sui prossimi periodi (rifit su tutta la storia)

In [6]:
N_FUTURE = 4
final_model = ARIMA(serie, order=(1, 1, 0))
final_fit = final_model.fit()
future_forecast = final_fit.get_forecast(steps=N_FUTURE)
future_mean = future_forecast.predicted_mean
future_ci = future_forecast.conf_int(alpha=0.2)

storico_e_previsione = pd.concat([serie.rename('storico'), future_mean.rename('previsione')])
fig = px.line(storico_e_previsione, title=f'{INDICATORE} - {TERRITORIO}: storico + previsione a {N_FUTURE} anni', markers=True)
fig

## (Opzionale) Scrivere le previsioni in `gold.forecast`

Così la dashboard Streamlit può mostrarle senza rieseguire il notebook. Ripeti il ciclo per tutte le combinazioni indicatore/territorio che ti interessano.

In [7]:
risultati = future_mean.reset_index()
risultati.columns = ['periodo', 'valore_previsto']
risultati['periodo'] = risultati['periodo'].dt.year.astype(str)
risultati['indicatore'] = INDICATORE
risultati['territorio'] = TERRITORIO

# risultati.to_sql('forecast', engine, schema='gold', if_exists='append', index=False)
risultati

,periodo,valore_previsto,indicatore,territorio
0,2026,2.780510,tasso_disoccupazione,Lombardia
1,2027,2.711688,tasso_disoccupazione,Lombardia
2,2028,2.690108,tasso_disoccupazione,Lombardia
3,2029,2.683341,tasso_disoccupazione,Lombardia
